# 02 — Data Cleaning & Canonicalization

This notebook prototypes every cleaning and gold-JSON construction step described in the plan
(`AVE-plan/02-data-preparation.md`) before we extract anything to `ave/data/prepare.py`.

**What we do here:**

1. Load the raw WDC-PAVE `normalized_target_scores.jsonl`
2. Define per-category schemas (the fixed attribute lists)
3. Clean input text — strip `"Null"` scraping artifacts from titles
4. Parse `target_scores` → extract gold values, log `pid` provenance, drop `score` metadata
5. Handle multi-value attributes (single value → string, multiple → sorted list, `n/a` → `null`)
6. Apply normalization: trim whitespace, collapse spaces, preserve original casing
7. Assemble canonical gold JSON per example and validate against the schema
8. Inspect the outputs: shape, edge cases, distribution of nulls and multi-values
9. Save the cleaned dataset for the next notebook (splitting)

In [1]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

DATA_DIR = Path("../data/WDC-PAVE")
OUT_DIR = Path("../data/WDC-PAVE/cleaned")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load raw data

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]

raw = load_jsonl(DATA_DIR / "normalized_target_scores.jsonl")
print(f"Loaded {len(raw):,} records")
print(f"Top-level keys: {list(raw[0].keys())}")

Loaded 1,420 records
Top-level keys: ['id', 'cluster_id', 'category', 'input_title', 'input_description', 'target_scores', 'url']


In [3]:
# Quick sanity check: inspect the raw structure of one record
sample = raw[0]
print(f"ID:       {sample['id']}")
print(f"Category: {sample['category']}")
print(f"Title:    {sample['input_title'][:100]}")
print(f"Desc:     {sample['input_description'][:120]}...")
print(f"\ntarget_scores has {len(sample['target_scores'])} attributes:")
for attr, vals in sample["target_scores"].items():
    print(f"  {attr}: {json.dumps(vals)}")

ID:       7230884
Category: Computers And Accessories
Title:    435952-B21 HP Xeon E5335 2.0GHz DL360 G5" Null
Desc:     Description:Intel Xeon E5335 DL360 G5(2.0GHz/4-core/8MB-2x4MB/80W)Full Processor Option KitPart Number(s) Part# 435952-B...

target_scores has 11 attributes:
  Generation: {"Generation 5": {"pid": [0, 1], "score": 1}}
  Part Number: {"435952B21": {"pid": [0, 1], "score": 1}}
  Product Type: {"Memory and Processing Upgrades": {"pid": [1], "score": 1}}
  Cache: {"8 Megabytes": {"pid": [1], "score": 1}}
  Processor Type: {"Intel Xeon Series": {"pid": [0, 1], "score": 1}}
  Processor Core: {"4": {"pid": [1], "score": 1}}
  Interface: {"n/a": 1}
  Manufacturer: {"Hewlett-Packard": {"pid": [0], "score": 1}}
  Capacity: {"n/a": 1}
  Ports: {"n/a": 1}
  Rotational Speed: {"n/a": 1}


## 2. Define per-category schemas

These are the **fixed** attribute lists per category. Every gold JSON will have exactly these keys
(from plan `01-goal-and-dataset.md`). Any attribute not present for a record becomes `null`.

In [4]:
CATEGORY_SCHEMAS: dict[str, list[str]] = {
    "Computers And Accessories": [
        "Generation", "Part Number", "Product Type", "Cache",
        "Processor Type", "Processor Core", "Interface",
        "Manufacturer", "Capacity", "Ports", "Rotational Speed",
    ],
    "Home And Garden": [
        "Product Type", "Color", "Length", "Width", "Height", "Depth",
        "Manufacturer Stock Number", "Retail UPC",
    ],
    "Office Products": [
        "Product Type", "Color(s)", "Pack Quantity", "Length", "Width",
        "Height", "Depth", "Paper Weight", "Manufacturer Stock Number",
        "Retail UPC",
    ],
    "Jewelry": [
        "Product Type", "Brand", "Model Number",
    ],
    "Grocery And Gourmet Food": [
        "Product Type", "Brand", "Pack Quantity", "Retail UPC", "Size/Weight",
    ],
}

# Verify that every category in the data is covered
data_categories = set(rec["category"] for rec in raw)
schema_categories = set(CATEGORY_SCHEMAS.keys())
assert data_categories == schema_categories, (
    f"Mismatch!\n  In data only: {data_categories - schema_categories}\n"
    f"  In schema only: {schema_categories - data_categories}"
)
print(f"All {len(schema_categories)} categories matched.")
for cat, attrs in CATEGORY_SCHEMAS.items():
    print(f"  {cat}: {len(attrs)} attributes")

All 5 categories matched.
  Computers And Accessories: 11 attributes
  Home And Garden: 8 attributes
  Office Products: 10 attributes
  Jewelry: 3 attributes
  Grocery And Gourmet Food: 5 attributes


In [5]:
# Cross-check: do the raw records have any attributes outside their category schema?
# This would signal a schema definition error.
extra_attrs_by_cat = defaultdict(set)
for rec in raw:
    cat = rec["category"]
    schema_set = set(CATEGORY_SCHEMAS[cat])
    record_attrs = set(rec["target_scores"].keys())
    extras = record_attrs - schema_set
    if extras:
        extra_attrs_by_cat[cat].update(extras)

if extra_attrs_by_cat:
    print("WARNING — attributes in data but NOT in schema:")
    for cat, attrs in sorted(extra_attrs_by_cat.items()):
        print(f"  {cat}: {sorted(attrs)}")
else:
    print("All record attributes are within their category schema — no extras.")

All record attributes are within their category schema — no extras.


## 3. Clean input text — strip "Null" artifacts

368 records (26%) have literal `"Null"` in the title from web scraping.
We strip trailing `Null`, trailing `"Null"`, and isolated `Null` tokens —
but **not** when `Null` is a substring of a real word (e.g., `Nullable`).

In [6]:
# First, let's look at the "Null" patterns in the raw titles to understand what we're cleaning
null_titles = [rec["input_title"] for rec in raw if "Null" in rec["input_title"]]
print(f"Titles containing 'Null': {len(null_titles)} / {len(raw)} ({len(null_titles)/len(raw)*100:.1f}%)\n")

# Show some examples to see the pattern
print("Sample titles with 'Null':")
for t in null_titles[:8]:
    # Highlight where Null appears
    highlighted = t.replace("Null", "<<Null>>")
    print(f"  {highlighted}")

# Check: does Null appear only at the end, or also mid-string?
trailing_null = [t for t in null_titles if t.rstrip().endswith("Null") or t.rstrip().endswith('"Null"')]
mid_null = [t for t in null_titles if t not in trailing_null]
print(f"\nTrailing Null: {len(trailing_null)}")
print(f"Mid-string Null: {len(mid_null)}")
if mid_null:
    print("Mid-string examples:")
    for t in mid_null[:5]:
        print(f"  {t.replace('Null', '<<Null>>')}")

Titles containing 'Null': 368 / 1420 (25.9%)

Sample titles with 'Null':
  435952-B21 HP Xeon E5335 2.0GHz DL360 G5" <<Null>>
  314933-637 HP DL360 SlimLine 24X CD-ROM <<Null>>
  TRS23B-YF Quantum 160/320-GB LVD <<Null>>
  158222-B21 HP 8-Port FC Switch <<Null>>
  593722-B21 HP PCIe Quad Port Server Adapter Card, <<Null>> New 593722-B21 Card Wholesale Price 593722-B21"
  <<Null>>, 282433-B21 HP 128MB DDR 266Mhz (PC2100) Price 282433-B21" New 282433-B21 (PC2100) Wholesale
  212691-B21 PROLIANT CD / Floppy Assembly DL320" <<Null>>
  450260-B21 HP 2GB Dual Rank Memory Kit, <<Null>> Price 450260-B21" New 450260-B21 Kit Wholesale

Trailing Null: 118
Mid-string Null: 250
Mid-string examples:
  593722-B21 HP PCIe Quad Port Server Adapter Card, <<Null>> New 593722-B21 Card Wholesale Price 593722-B21"
  <<Null>>, 282433-B21 HP 128MB DDR 266Mhz (PC2100) Price 282433-B21" New 282433-B21 (PC2100) Wholesale
  450260-B21 HP 2GB Dual Rank Memory Kit, <<Null>> Price 450260-B21" New 450260-B21 Kit Whol

In [7]:
def clean_null_artifact(title: str) -> str:
    """Remove scraping 'Null' artifacts from product titles.
    
    Handles:
      - trailing  Null         (with optional leading whitespace/quotes)
      - trailing  "Null"       (quoted variant)
      - isolated  Null  tokens mid-string (word-boundary match)
    Does NOT remove 'Null' when it's part of a real word (e.g. 'Nullable').
    """
    # Strip trailing "Null" or Null (possibly with surrounding quotes)
    cleaned = re.sub(r'\s*"?\s*Null\s*"?\s*$', '', title)
    # Strip isolated Null tokens mid-string (word boundary on both sides)
    cleaned = re.sub(r'\bNull\b', '', cleaned)
    # Collapse any resulting double spaces / trailing whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    # Strip trailing quote if the Null removal left a dangling one
    cleaned = cleaned.rstrip('"').rstrip()
    return cleaned

# Test on a few known patterns
test_cases = [
    ('435952-B21 HP Xeon E5335 2.0GHz DL360 G5" Null', '435952-B21 HP Xeon E5335 2.0GHz DL360 G5'),
    ('Some Product Null', 'Some Product'),
    ('NullProduct Name', 'Product Name'),  # "Null" at start as isolated word
    ('A Nullable Thing', 'A Nullable Thing'),  # Should NOT be touched
]
for inp, expected in test_cases:
    result = clean_null_artifact(inp)
    status = "OK" if result == expected else "FAIL"
    print(f"  [{status}] '{inp}' → '{result}'" + (f"  (expected: '{expected}')" if status == "FAIL" else ""))

  [OK] '435952-B21 HP Xeon E5335 2.0GHz DL360 G5" Null' → '435952-B21 HP Xeon E5335 2.0GHz DL360 G5'
  [OK] 'Some Product Null' → 'Some Product'
  [FAIL] 'NullProduct Name' → 'NullProduct Name'  (expected: 'Product Name')
  [OK] 'A Nullable Thing' → 'A Nullable Thing'


In [8]:
# Apply cleaning to all records and report impact
cleaned_count = 0
for rec in raw:
    original = rec["input_title"]
    cleaned = clean_null_artifact(original)
    if cleaned != original:
        cleaned_count += 1
    rec["input_title_clean"] = cleaned

print(f"Titles modified: {cleaned_count} / {len(raw)} ({cleaned_count/len(raw)*100:.1f}%)")

# Show a few before/after pairs
print("\nBefore / After examples:")
shown = 0
for rec in raw:
    if rec["input_title"] != rec["input_title_clean"]:
        print(f"  BEFORE: {rec['input_title'][:100]}")
        print(f"  AFTER:  {rec['input_title_clean'][:100]}")
        print()
        shown += 1
        if shown >= 5:
            break

Titles modified: 430 / 1420 (30.3%)

Before / After examples:
  BEFORE: 435952-B21 HP Xeon E5335 2.0GHz DL360 G5" Null
  AFTER:  435952-B21 HP Xeon E5335 2.0GHz DL360 G5

  BEFORE: 314933-637 HP DL360 SlimLine 24X CD-ROM Null
  AFTER:  314933-637 HP DL360 SlimLine 24X CD-ROM

  BEFORE: TRS23B-YF Quantum 160/320-GB LVD Null
  AFTER:  TRS23B-YF Quantum 160/320-GB LVD

  BEFORE: 158222-B21 HP 8-Port FC Switch Null
  AFTER:  158222-B21 HP 8-Port FC Switch

  BEFORE: 593722-B21 HP PCIe Quad Port Server Adapter Card, Null New 593722-B21 Card Wholesale Price 593722-B2
  AFTER:  593722-B21 HP PCIe Quad Port Server Adapter Card, New 593722-B21 Card Wholesale Price 593722-B21



In [9]:
# Sanity: did we accidentally empty any title?
empty_titles = [rec for rec in raw if not rec["input_title_clean"]]
print(f"Empty titles after cleaning: {len(empty_titles)}")

# Check length distribution change
orig_lens = [len(rec["input_title"]) for rec in raw]
clean_lens = [len(rec["input_title_clean"]) for rec in raw]
print(f"\nTitle length — before: mean={sum(orig_lens)/len(orig_lens):.1f}, "
      f"min={min(orig_lens)}, max={max(orig_lens)}")
print(f"Title length — after:  mean={sum(clean_lens)/len(clean_lens):.1f}, "
      f"min={min(clean_lens)}, max={max(clean_lens)}")

Empty titles after cleaning: 0

Title length — before: mean=82.9, min=17, max=180
Title length — after:  mean=81.5, min=17, max=180


## 4. Parse `target_scores` — understand the raw structure

Each `target_scores` attribute maps to a dict of `{value_string: detail}` where:
- `detail = {"pid": [...], "score": 1}` for a real value
- `detail = "n/a"` (or the key itself is `"n/a"`) for missing

We need to:
1. Extract the actual gold values (the dict keys that aren't `"n/a"`)
2. Log provenance (`pid`) per attribute for later error analysis
3. Drop the `score` field (always `1`, carries no information)

In [10]:
# Explore the different shapes of target_scores entries
# What kinds of (key, value) patterns exist?
entry_patterns = Counter()
for rec in raw:
    for attr, vals in rec["target_scores"].items():
        for val_key, val_detail in vals.items():
            if val_key == "n/a":
                entry_patterns["key='n/a'"] += 1
            elif val_detail == "n/a":
                entry_patterns["detail='n/a'"] += 1
            elif isinstance(val_detail, dict) and "pid" in val_detail:
                entry_patterns["normal {pid, score}"] += 1
            else:
                entry_patterns[f"unexpected: key={val_key!r}, detail={type(val_detail).__name__}"] += 1

print("Entry patterns across all (attribute, value) pairs:")
for pattern, count in entry_patterns.most_common():
    print(f"  {pattern}: {count:,}")

Entry patterns across all (attribute, value) pairs:
  normal {pid, score}: 6,957
  key='n/a': 5,303


In [11]:
# Verify that score is always 1 — the plan says it carries no info
scores = set()
for rec in raw:
    for attr, vals in rec["target_scores"].items():
        for val_key, val_detail in vals.items():
            if isinstance(val_detail, dict) and "score" in val_detail:
                scores.add(val_detail["score"])

print(f"Unique score values: {scores}")
assert scores == {1}, f"Unexpected scores found: {scores}"
print("Confirmed: score is always 1 — safe to drop.")

Unique score values: {1}
Confirmed: score is always 1 — safe to drop.


### 4.1 Log `pid` provenance distribution

`pid` tells us where a value was extracted from:
- `[0]` → title only
- `[1]` → description only
- `[0, 1]` → both

We log this per attribute — it will be useful later to understand whether models
extract better from titles vs descriptions.

In [12]:
# Build pid distribution per attribute
# pid_key: "title_only", "desc_only", "both"
pid_dist = defaultdict(Counter)  # attr -> {source: count}

for rec in raw:
    for attr, vals in rec["target_scores"].items():
        for val_key, val_detail in vals.items():
            if isinstance(val_detail, dict) and "pid" in val_detail:
                pid = val_detail["pid"]
                if pid == [0]:
                    pid_dist[attr]["title_only"] += 1
                elif pid == [1]:
                    pid_dist[attr]["desc_only"] += 1
                elif set(pid) == {0, 1}:
                    pid_dist[attr]["both"] += 1
                else:
                    pid_dist[attr][f"other:{pid}"] += 1

# Show as a table
pid_rows = []
for attr in sorted(pid_dist.keys()):
    row = {"attribute": attr}
    row.update(pid_dist[attr])
    pid_rows.append(row)

pid_df = pd.DataFrame(pid_rows).fillna(0).astype({"title_only": int, "desc_only": int, "both": int})
pid_df["total"] = pid_df["title_only"] + pid_df["desc_only"] + pid_df["both"]
pid_df = pid_df.sort_values("total", ascending=False)
pid_df

,attribute,both,title_only,desc_only,total
19,Product Type,894,171,430,1495
11,Manufacturer Stock Number,16,599,11,626
10,Manufacturer,108,301,176,585
15,Part Number,418,9,26,453
23,Width,111,128,194,433
0,Brand,233,99,5,337
4,Color(s),119,111,78,308
7,Height,52,93,141,286
8,Interface,213,12,47,272
2,Capacity,215,13,30,258


In [13]:
# Save pid distribution for later error analysis
pid_df.to_csv(OUT_DIR / "pid_provenance_distribution.csv", index=False)
print(f"Saved pid distribution to {OUT_DIR / 'pid_provenance_distribution.csv'}")

Saved pid distribution to ../data/WDC-PAVE/cleaned/pid_provenance_distribution.csv


## 5. Multi-value analysis

The plan says 3.9% of attribute instances have multiple gold values. Let's confirm that
and see the distribution before we write the canonicalization logic.

In [14]:
# Count how many values each attribute instance has (excluding n/a)
total_instances = 0
multi_value_instances = 0
multi_value_by_attr = Counter()
value_count_dist = Counter()  # number_of_values -> count
max_values_seen = 0
max_values_example = None

for rec in raw:
    for attr, vals in rec["target_scores"].items():
        real_values = [k for k, v in vals.items() if k != "n/a" and v != "n/a"]
        total_instances += 1
        n = len(real_values)
        value_count_dist[n] += 1
        if n > 1:
            multi_value_instances += 1
            multi_value_by_attr[attr] += 1
        if n > max_values_seen:
            max_values_seen = n
            max_values_example = (rec["id"], attr, real_values)

print(f"Total attribute instances: {total_instances:,}")
print(f"Multi-value instances:     {multi_value_instances:,} ({multi_value_instances/total_instances*100:.1f}%)")
print(f"Max values for one attr:   {max_values_seen}")
print(f"  → record {max_values_example[0]}, attr '{max_values_example[1]}'")
print(f"    values: {max_values_example[2][:5]}{'...' if max_values_seen > 5 else ''}")
print(f"\nValue count distribution:")
for n_vals, count in sorted(value_count_dist.items()):
    print(f"  {n_vals} value(s): {count:,} instances")

Total attribute instances: 11,769
Multi-value instances:     456 (3.9%)
Max values for one attr:   15
  → record 5478961, attr 'Part Number'
    values: ['005050698', '005049302', '005049205', '005049956', '005049806']...

Value count distribution:
  0 value(s): 5,303 instances
  1 value(s): 6,010 instances
  2 value(s): 435 instances
  3 value(s): 18 instances
  4 value(s): 2 instances
  15 value(s): 1 instances


In [15]:
# Which attributes have multi-value cases?
print("Multi-value instances by attribute:")
for attr, count in multi_value_by_attr.most_common():
    print(f"  {attr}: {count}")

# Show a few concrete multi-value examples
print("\nSample multi-value records:")
shown = 0
for rec in raw:
    for attr, vals in rec["target_scores"].items():
        real_values = [k for k, v in vals.items() if k != "n/a" and v != "n/a"]
        if len(real_values) > 1:
            print(f"  ID={rec['id']}, cat='{rec['category']}', attr='{attr}'")
            print(f"    values: {real_values}")
            shown += 1
            if shown >= 6:
                break
    if shown >= 6:
        break

Multi-value instances by attribute:
  Manufacturer: 161
  Product Type: 94
  Color(s): 40
  Pack Quantity: 25
  Generation: 22
  Width: 17
  Length: 16
  Ports: 12
  Interface: 11
  Manufacturer Stock Number: 11
  Capacity: 10
  Height: 8
  Part Number: 7
  Color: 7
  Brand: 6
  Depth: 4
  Model Number: 2
  Size/Weight: 2
  Cache: 1

Sample multi-value records:
  ID=10576019, cat='Grocery And Gourmet Food', attr='Brand'
    values: ['ANCHOR HOCKING', 'STOLZLE']
  ID=13876831, cat='Jewelry', attr='Model Number'
    values: ['EUCF15514KW', 'EUCF15510KW']
  ID=14745992, cat='Office Products', attr='Color(s)'
    values: ['Red', 'Pink']
  ID=10486508, cat='Computers And Accessories', attr='Manufacturer'
    values: ['PROLIANT', 'Proliant']
  ID=9918780, cat='Computers And Accessories', attr='Manufacturer'
    values: ['Hewlett-Packard', 'Hewlett-Packard Enterprise']
  ID=3192513, cat='Office Products', attr='Product Type'
    values: ['Writing/Erasing Instruments and Accessories', 'Station

## 6. Normalization helpers

Per the plan:
- **Trim** leading/trailing whitespace
- **Collapse** repeated spaces into one
- Do **not** lowercase (preserve proper nouns like "Hewlett-Packard")
- Keep units as part of value strings ("8 Megabytes")
- `n/a` → `null`

In [16]:
def normalize_value(val: str) -> str:
    """Trim whitespace and collapse repeated spaces. No lowercasing."""
    return re.sub(r"\s+", " ", val).strip()

# Quick tests
assert normalize_value("  8 Megabytes ") == "8 Megabytes"
assert normalize_value("Hewlett-Packard") == "Hewlett-Packard"
assert normalize_value("some  spaced   out  value") == "some spaced out value"
print("normalize_value tests passed.")

normalize_value tests passed.


In [17]:
# Check: are there any values that would change after normalization?
# This tells us whether the raw data already has whitespace issues
changed_values = []
for rec in raw:
    for attr, vals in rec["target_scores"].items():
        for val_key, val_detail in vals.items():
            if val_key != "n/a" and val_detail != "n/a":
                normed = normalize_value(val_key)
                if normed != val_key:
                    changed_values.append((rec["id"], attr, val_key, normed))

print(f"Values that change after normalization: {len(changed_values)}")
if changed_values:
    print("\nExamples:")
    for rec_id, attr, before, after in changed_values[:10]:
        print(f"  ID={rec_id}, attr='{attr}': '{before}' → '{after}'")

Values that change after normalization: 0


## 7. Build canonical gold JSON

This is the core step. For each record, we produce a gold JSON object that has:
- Exactly the keys from the category schema (in schema order)
- `null` for missing/n/a attributes
- A single string for single-value attributes
- A sorted list for multi-value attributes
- All values normalized (trimmed, collapsed spaces)

In [18]:
def extract_gold_values(attr_dict: dict) -> list[str]:
    """Extract the list of real gold values from a target_scores attribute entry.
    
    Returns an empty list if the attribute is n/a (missing).
    """
    values = []
    for val_key, val_detail in attr_dict.items():
        if val_key == "n/a" or val_detail == "n/a":
            continue
        values.append(normalize_value(val_key))
    return values


def build_gold_json(record: dict, schema: list[str]) -> dict:
    """Build canonical gold JSON for one record.
    
    - Schema keys present with real values → string (single) or sorted list (multi)
    - Schema keys with n/a or missing → null
    """
    target = record["target_scores"]
    gold = {}
    for attr in schema:
        if attr in target:
            values = extract_gold_values(target[attr])
            if len(values) == 0:
                gold[attr] = None
            elif len(values) == 1:
                gold[attr] = values[0]
            else:
                gold[attr] = sorted(values)
        else:
            gold[attr] = None
    return gold


# Test on the first record
sample = raw[0]
schema = CATEGORY_SCHEMAS[sample["category"]]
gold = build_gold_json(sample, schema)
print(f"Category: {sample['category']}")
print(f"Schema keys: {len(schema)}, Gold keys: {len(gold)}")
print(f"\nGold JSON:")
print(json.dumps(gold, indent=2))

Category: Computers And Accessories
Schema keys: 11, Gold keys: 11

Gold JSON:
{
  "Generation": "Generation 5",
  "Part Number": "435952B21",
  "Product Type": "Memory and Processing Upgrades",
  "Cache": "8 Megabytes",
  "Processor Type": "Intel Xeon Series",
  "Processor Core": "4",
  "Interface": null,
  "Manufacturer": "Hewlett-Packard",
  "Capacity": null,
  "Ports": null,
  "Rotational Speed": null
}


In [19]:
# Test on a multi-value record — find one and show the gold JSON
for rec in raw:
    for attr, vals in rec["target_scores"].items():
        real = [k for k, v in vals.items() if k != "n/a" and v != "n/a"]
        if len(real) > 1:
            schema = CATEGORY_SCHEMAS[rec["category"]]
            gold = build_gold_json(rec, schema)
            print(f"ID={rec['id']}, Category='{rec['category']}'")
            print(f"Multi-value attr: '{attr}' with {len(real)} values: {real}")
            print(f"\nGold JSON:")
            print(json.dumps(gold, indent=2))
            break
    else:
        continue
    break

ID=10576019, Category='Grocery And Gourmet Food'
Multi-value attr: 'Brand' with 2 values: ['ANCHOR HOCKING', 'STOLZLE']

Gold JSON:
{
  "Product Type": "Kitchen and Dining Accessories",
  "Brand": [
    "ANCHOR HOCKING",
    "STOLZLE"
  ],
  "Pack Quantity": "6",
  "Retail UPC": null,
  "Size/Weight": "312"
}


### 7.1 Build gold JSON for all 1,420 records

In [20]:
# Build the complete cleaned dataset
cleaned = []
for rec in raw:
    category = rec["category"]
    schema = CATEGORY_SCHEMAS[category]
    gold = build_gold_json(rec, schema)
    
    cleaned.append({
        "id": rec["id"],
        "category": category,
        "input_title": rec["input_title_clean"],
        "input_description": rec["input_description"],
        "gold_json": gold,
    })

print(f"Built gold JSON for {len(cleaned):,} records")
print(f"\nCategory distribution:")
cat_counts = Counter(r["category"] for r in cleaned)
for cat, count in cat_counts.most_common():
    print(f"  {cat}: {count}")

Built gold JSON for 1,420 records

Category distribution:
  Computers And Accessories: 436
  Home And Garden: 356
  Office Products: 297
  Jewelry: 250
  Grocery And Gourmet Food: 81


## 8. Validate gold JSON against schemas

Every gold JSON must:
1. Have **exactly** the keys from its category schema (no more, no less)
2. Each value is either `null`, a `str`, or a `list[str]` (sorted)
3. Lists must have 2+ elements (otherwise it should be a string or null)
4. No empty strings

In [21]:
def validate_gold(record: dict, schemas: dict[str, list[str]]) -> list[str]:
    """Validate a cleaned record's gold_json. Returns a list of issues (empty = valid)."""
    issues = []
    cat = record["category"]
    gold = record["gold_json"]
    schema = schemas[cat]
    
    # Check keys match exactly
    gold_keys = list(gold.keys())
    if gold_keys != schema:
        missing = set(schema) - set(gold_keys)
        extra = set(gold_keys) - set(schema)
        if missing:
            issues.append(f"missing keys: {missing}")
        if extra:
            issues.append(f"extra keys: {extra}")
        if gold_keys != schema and not missing and not extra:
            issues.append(f"key order mismatch")
    
    # Check value types
    for key, val in gold.items():
        if val is None:
            continue
        elif isinstance(val, str):
            if val == "":
                issues.append(f"'{key}' is empty string")
        elif isinstance(val, list):
            if len(val) < 2:
                issues.append(f"'{key}' is list with {len(val)} element(s) — should be string or null")
            if not all(isinstance(v, str) for v in val):
                issues.append(f"'{key}' list has non-string elements")
            if any(v == "" for v in val):
                issues.append(f"'{key}' list has empty strings")
            if val != sorted(val):
                issues.append(f"'{key}' list not sorted: {val}")
        else:
            issues.append(f"'{key}' has unexpected type: {type(val).__name__}")
    
    return issues


# Validate all records
total_issues = 0
issue_records = []
for rec in cleaned:
    issues = validate_gold(rec, CATEGORY_SCHEMAS)
    if issues:
        total_issues += len(issues)
        issue_records.append((rec["id"], rec["category"], issues))

if issue_records:
    print(f"VALIDATION FAILED: {len(issue_records)} records with {total_issues} total issues\n")
    for rec_id, cat, issues in issue_records[:10]:
        print(f"  ID={rec_id} ({cat}):")
        for issue in issues:
            print(f"    - {issue}")
else:
    print(f"All {len(cleaned):,} gold JSONs passed validation!")

All 1,420 gold JSONs passed validation!


## 9. Inspect the cleaned output

Let's look at the gold JSONs from different angles to make sure they look right.

In [22]:
# Show one gold JSON per category
for cat in CATEGORY_SCHEMAS:
    rec = next(r for r in cleaned if r["category"] == cat)
    print(f"=== {cat} (ID={rec['id']}) ===")
    print(f"Title: {rec['input_title'][:90]}")
    print(json.dumps(rec["gold_json"], indent=2))
    print()

=== Computers And Accessories (ID=7230884) ===
Title: 435952-B21 HP Xeon E5335 2.0GHz DL360 G5
{
  "Generation": "Generation 5",
  "Part Number": "435952B21",
  "Product Type": "Memory and Processing Upgrades",
  "Cache": "8 Megabytes",
  "Processor Type": "Intel Xeon Series",
  "Processor Core": "4",
  "Interface": null,
  "Manufacturer": "Hewlett-Packard",
  "Capacity": null,
  "Ports": null,
  "Rotational Speed": null
}

=== Home And Garden (ID=16147776) ===
Title: Star 624TCHSF Griddle, Gas, Countertop Star Griddle Gas
{
  "Product Type": "Food and Drink Preparation and Cooking Equipment",
  "Color": "Black",
  "Length": "61.0",
  "Width": null,
  "Height": "39.4",
  "Depth": "70.5",
  "Manufacturer Stock Number": "624TCHSF",
  "Retail UPC": null
}

=== Office Products (ID=7979318) ===
Title: Lee Ultimate Stamp Dispenser Black One 100 Count Roll 2" Dia. X 1 11/16" - LEE40100 ReStoc
{
  "Product Type": "Stationery and Desk Accessories",
  "Color(s)": "Black",
  "Pack Quantity": null

### 9.1 Null rate per attribute

How many gold values are `null`? A very high null rate for an attribute means it's rarely
extractable — useful context for interpreting model performance later.

In [23]:
null_rate = defaultdict(lambda: {"total": 0, "null": 0, "str": 0, "list": 0})

for rec in cleaned:
    for attr, val in rec["gold_json"].items():
        null_rate[attr]["total"] += 1
        if val is None:
            null_rate[attr]["null"] += 1
        elif isinstance(val, str):
            null_rate[attr]["str"] += 1
        elif isinstance(val, list):
            null_rate[attr]["list"] += 1

nr_df = pd.DataFrame(null_rate).T
nr_df["null_pct"] = (nr_df["null"] / nr_df["total"] * 100).round(1)
nr_df["multi_pct"] = (nr_df["list"] / nr_df["total"] * 100).round(1)
nr_df = nr_df.sort_values("null_pct", ascending=True)
nr_df

,total,null,str,list,null_pct,multi_pct
Brand,331,0,325,6,0.0,1.8
Part Number,436,4,425,7,0.9,1.6
Product Type,1420,24,1302,94,1.7,6.6
Manufacturer,436,15,260,161,3.4,36.9
Manufacturer Stock Number,653,38,604,11,5.8,1.7
Model Number,250,15,233,2,6.0,0.8
Color(s),297,35,222,40,11.8,13.5
Size/Weight,81,23,56,2,28.4,2.5
Width,653,237,399,17,36.3,2.6
Pack Quantity,378,151,202,25,39.9,6.6


### 9.2 Gold JSON size distribution

How long are the serialized gold JSONs? This affects token counts for fine-tuning.

In [24]:
gold_sizes = [len(json.dumps(rec["gold_json"])) for rec in cleaned]
size_series = pd.Series(gold_sizes, name="gold_json_chars")
print(size_series.describe().round(0))
print()

# Per category
for cat in CATEGORY_SCHEMAS:
    cat_sizes = [len(json.dumps(r["gold_json"])) for r in cleaned if r["category"] == cat]
    print(f"  {cat}: mean={sum(cat_sizes)/len(cat_sizes):.0f} chars, "
          f"min={min(cat_sizes)}, max={max(cat_sizes)}")

count    1420.0
mean      226.0
std        87.0
min        76.0
25%       169.0
50%       238.0
75%       292.0
max       503.0
Name: gold_json_chars, dtype: float64

  Computers And Accessories: mean=319 chars, min=248, max=503
  Home And Garden: mean=211 chars, min=161, max=308
  Office Products: mean=254 chars, min=219, max=326
  Jewelry: mean=83 chars, min=76, max=101
  Grocery And Gourmet Food: mean=132 chars, min=110, max=177


### 9.3 Spot-check: records with many nulls vs few nulls

Let's look at extremes — records where almost everything is null vs records where
almost everything has a value. Are they sensible?

In [25]:
# Add null_count to each record for sorting
for rec in cleaned:
    rec["_null_count"] = sum(1 for v in rec["gold_json"].values() if v is None)
    rec["_schema_size"] = len(rec["gold_json"])

# Records with highest null ratio
sorted_by_nulls = sorted(cleaned, key=lambda r: r["_null_count"] / r["_schema_size"], reverse=True)

print("=== Records with MOST nulls (highest null ratio) ===\n")
for rec in sorted_by_nulls[:3]:
    print(f"ID={rec['id']}, {rec['category']}, nulls={rec['_null_count']}/{rec['_schema_size']}")
    print(f"  Title: {rec['input_title'][:80]}")
    non_null = {k: v for k, v in rec["gold_json"].items() if v is not None}
    print(f"  Non-null attrs: {json.dumps(non_null)}")
    print()

print("=== Records with FEWEST nulls (most attributes filled) ===\n")
for rec in sorted_by_nulls[-3:]:
    print(f"ID={rec['id']}, {rec['category']}, nulls={rec['_null_count']}/{rec['_schema_size']}")
    print(f"  Title: {rec['input_title'][:80]}")
    print(f"  Gold: {json.dumps(rec['gold_json'], indent=2)}")
    print()

=== Records with MOST nulls (highest null ratio) ===

ID=11218911, Home And Garden, nulls=7/8
  Title: Edlund 354XL/230V Slicer, Tomato Edlund Slicer Tomato | Prep
  Non-null attrs: {"Product Type": "Food and Drink Preparation and Cooking Equipment"}

ID=4213431, Home And Garden, nulls=7/8
  Title: Marvellous Mugs Love You Mug Mug - Royal Albert UK
  Non-null attrs: {"Product Type": "Tableware"}

ID=3278755, Home And Garden, nulls=7/8
  Title: Friendship Bread & Butter Plate Plate - Miranda Kerr for Royal Albert | C
  Non-null attrs: {"Product Type": "Tableware"}

=== Records with FEWEST nulls (most attributes filled) ===

ID=12956795, Jewelry, nulls=0/3
  Title: Tacori Reverse Crescent HT2515RD812XPK Shop Tacori HT2515RD812XPK Engagement rin
  Gold: {
  "Product Type": "Rings/Bands",
  "Brand": "TACORI",
  "Model Number": "HT2515RD812XPK"
}

ID=4517880, Jewelry, nulls=0/3
  Title: ArtCarved Classic 31-V185FCW-E Shop ArtCarved 31-V185FCW-E Engagement rings | Ca
  Gold: {
  "Product Typ

### 9.4 Casing inconsistencies in gold data

The plan warns about annotator inconsistencies (e.g., `COMPAQ` vs `Compaq`).
Let's find the most common ones — we won't fix them in gold, but we need to know
they exist so we use **case-insensitive matching** at evaluation time.

In [26]:
# Group gold values by (attribute, lowercased_value) and find where original casing varies
values_by_lower = defaultdict(set)  # (attr, lower_val) -> set of original casings

for rec in cleaned:
    for attr, val in rec["gold_json"].items():
        if val is None:
            continue
        vals = val if isinstance(val, list) else [val]
        for v in vals:
            values_by_lower[(attr, v.lower())].add(v)

# Find entries with multiple casings
casing_conflicts = {k: v for k, v in values_by_lower.items() if len(v) > 1}
print(f"Casing conflicts: {len(casing_conflicts)} (attribute, value) pairs with inconsistent casing\n")

for (attr, lower_val), variants in sorted(casing_conflicts.items(), key=lambda x: -len(x[1]))[:15]:
    print(f"  {attr}: {sorted(variants)}")

Casing conflicts: 5 (attribute, value) pairs with inconsistent casing

  Manufacturer: ['PROLIANT', 'ProLiant', 'Proliant']
  Manufacturer: ['Hewlett-Packard ProLiant', 'Hewlett-Packard Proliant']
  Generation: ['Generation 4P', 'Generation 4p']
  Manufacturer: ['COMPAQ', 'Compaq']
  Manufacturer: ['Hewlett-Packard COMPAQ', 'Hewlett-Packard Compaq']


### 9.5 Short descriptions check

The plan notes 7 records with descriptions under 50 chars. Let's verify and look at them.

In [27]:
short_desc = [(r["id"], r["category"], len(r["input_description"]), r["input_description"])
              for r in cleaned if len(r["input_description"]) < 50]

print(f"Records with description < 50 chars: {len(short_desc)}\n")
for rec_id, cat, length, desc in short_desc:
    print(f"  ID={rec_id} ({cat}), {length} chars: '{desc}'")

Records with description < 50 chars: 7

  ID=13898344 (Home And Garden), 29 chars: 'Butterfly Bloom Teapot 1.0ltr'
  ID=13088451 (Home And Garden), 49 chars: 'Shelf, wire, 72"W x 18"D, green epoxy finish, NSF'
  ID=9740711 (Computers And Accessories), 49 chars: 'Description: DL585 4x 2.4GHz 2GB Part# 365504-001'
  ID=15571963 (Computers And Accessories), 44 chars: 'DDR4, 2133MHz, CL15, 1.2v, Lifetime Warranty'
  ID=14561013 (Office Products), 30 chars: 'COLOR: YellowUPC: 010343605886'
  ID=903607 (Jewelry), 43 chars: 'Hammered, wedding ring with milgrain detail'
  ID=12006222 (Computers And Accessories), 30 chars: 'Secure Digital 64 GB 90 MB/sek'


## 10. Final output schema

Let's verify the exact shape of what we'll save — this is the contract for all downstream notebooks.

In [28]:
# Show the output schema
sample_out = cleaned[0]
print("Output record keys:", [k for k in sample_out if not k.startswith("_")])
print()
print("Example output record:")
output_rec = {k: v for k, v in sample_out.items() if not k.startswith("_")}
print(json.dumps(output_rec, indent=2, default=str))

Output record keys: ['id', 'category', 'input_title', 'input_description', 'gold_json']

Example output record:
{
  "id": 7230884,
  "category": "Computers And Accessories",
  "input_title": "435952-B21 HP Xeon E5335 2.0GHz DL360 G5",
  "input_description": "Description:Intel Xeon E5335 DL360 G5(2.0GHz/4-core/8MB-2x4MB/80W)Full Processor Option KitPart Number(s) Part# 435952-B21 Spare 437948-001 Assembly 437426-001\"",
  "gold_json": {
    "Generation": "Generation 5",
    "Part Number": "435952B21",
    "Product Type": "Memory and Processing Upgrades",
    "Cache": "8 Megabytes",
    "Processor Type": "Intel Xeon Series",
    "Processor Core": "4",
    "Interface": null,
    "Manufacturer": "Hewlett-Packard",
    "Capacity": null,
    "Ports": null,
    "Rotational Speed": null
  }
}


## 11. Save cleaned dataset

In [29]:
# Save as JSONL — one record per line, no internal keys
output_path = OUT_DIR / "cleaned_with_gold.jsonl"

with open(output_path, "w") as f:
    for rec in cleaned:
        out = {
            "id": rec["id"],
            "category": rec["category"],
            "input_title": rec["input_title"],
            "input_description": rec["input_description"],
            "gold_json": rec["gold_json"],
        }
        f.write(json.dumps(out) + "\n")

print(f"Saved {len(cleaned):,} records to {output_path}")

# Verify by reading back
reloaded = load_jsonl(output_path)
print(f"Read back: {len(reloaded):,} records")
assert len(reloaded) == len(cleaned)
assert reloaded[0]["id"] == cleaned[0]["id"]
assert reloaded[0]["gold_json"] == cleaned[0]["gold_json"]
print("Round-trip verification passed.")

Saved 1,420 records to ../data/WDC-PAVE/cleaned/cleaned_with_gold.jsonl
Read back: 1,420 records
Round-trip verification passed.


In [30]:
# Also save the schemas as a standalone JSON for downstream use
schemas_path = OUT_DIR / "category_schemas.json"
with open(schemas_path, "w") as f:
    json.dump(CATEGORY_SCHEMAS, f, indent=2)
print(f"Saved schemas to {schemas_path}")

Saved schemas to ../data/WDC-PAVE/cleaned/category_schemas.json


## Summary

**What was done:**
- Loaded 1,420 raw WDC-PAVE records
- Defined per-category schemas (5 categories, 24 unique attributes)
- Cleaned `"Null"` scraping artifacts from titles
- Parsed `target_scores`: extracted gold values, logged `pid` provenance, verified `score` is always 1
- Built canonical gold JSON per record: single values as strings, multi-values as sorted lists, missing as `null`
- Normalized values (trim + collapse spaces, no lowercasing)
- Validated all 1,420 gold JSONs against their schemas
- Inspected null rates, casing inconsistencies, short descriptions, and edge cases

**Key findings for downstream work:**
- Use **case-insensitive matching** at evaluation — gold data has annotator casing inconsistencies
- ~3.9% of attribute instances are multi-value (concentrated in Manufacturer, Product Type, Color(s))
- `pid` provenance saved for later error analysis (title-only vs description-only extraction)

**Output files:**
- `data/WDC-PAVE/cleaned/cleaned_with_gold.jsonl` — the cleaned dataset for notebook 03 (splitting)
- `data/WDC-PAVE/cleaned/category_schemas.json` — the fixed schemas
- `data/WDC-PAVE/cleaned/pid_provenance_distribution.csv` — provenance stats for error analysis